# 📊 Lab 05: RAG Evaluation with RAGAS - 系統評估

## 學習目標
在本實驗中，您將學習：
1. **RAGAS 框架介紹** - RAG 評估的標準化方法
2. **核心評估指標** - Faithfulness, Relevancy, Precision, Recall
3. **建立測試資料集** - 生成問答對用於評估
4. **系統評估** - 評估並比較不同 RAG 配置
5. **結果分析** - 解讀評估結果並優化系統

## 為什麼需要評估 RAG 系統？
- 量化系統品質，而非主觀判斷
- 比較不同配置的效果
- 識別系統弱點並改進

---

## 📦 Part 1: 環境設置

# 安裝必要套件
!pip install --quiet langchain langchain-community langchain-text-splitters
!pip install --quiet chromadb sentence-transformers langchain-huggingface langchain-ollama
!pip install --quiet ragas datasets pandas matplotlib
!pip install --quiet requests numpy


In [ ]:
# 準備 RAG 系統
import os
os.environ["OLLAMA_HOST"] = "http://host.docker.internal:11434"

# 設定本地 HuggingFace cache 目錄
os.environ["HF_HOME"] = "./hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
import requests
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA


# 下載範例文件
os.makedirs('data', exist_ok=True)
url = "https://www.gutenberg.org/files/11/11-0.txt"
output_path = "data/alice_in_wonderland.txt"

if not os.path.exists(output_path):
    response = requests.get(url)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(response.text)

# 載入並分割
loader = TextLoader(output_path, encoding='utf-8')
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# 初始化
embeddings = HuggingFaceEmbeddings(
    #model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_name="voyageai/voyage-4-nano",
    cache_folder="./hf_cache",
    model_kwargs={"device": "cuda"},
)
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 建立 RAG Chain
#MODEL_NAME = "llama3.2"
MODEL_NAME = "qwen3:1.7b" 
llm = OllamaLLM(model=MODEL_NAME)

template = """根據以下上下文回答問題。如果無法從上下文找到答案，請說明。

上下文: {context}
問題: {question}

回答:"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])

rag_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever,
    return_source_documents=True, chain_type_kwargs={"prompt": prompt}
)

print(f"✅ RAG 系統已準備就緒！(共 {len(chunks)} chunks)")



Loading weights: 100%|██████████| 134/134 [00:00<00:00, 2666.15it/s]
Qwen3Model LOAD REPORT from: voyageai/voyage-4-nano
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ RAG 系統已準備就緒！(共 411 chunks)


In [ ]:
---
## 📋 Part 2: RAGAS 核心指標介紹

RAGAS 提供四個核心評估指標：
- **Faithfulness**: 回答是否忠於檢索到的上下文
- **Answer Relevancy**: 回答與問題的相關程度
- **Context Precision**: 檢索內容的精確度
- **Context Recall**: 檢索內容的召回率

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.1 MB/s eta 0:00:00
   ━━

In [13]:
# 建立測試資料集
test_questions = [
    "How did Alice fall into Wonderland?",
    "Who is the Queen of Hearts?",
    "What happened at the Mad Tea Party?",
    "What is special about the Cheshire Cat?",
    "How does Alice change size in the story?",
]

# 對應的標準答案 (Ground Truth)
ground_truths = [
    "Alice fell down a rabbit hole while following a white rabbit.",
    "The Queen of Hearts is a tyrannical ruler who orders executions.",
    "Alice attended a tea party with the Mad Hatter and March Hare.",
    "The Cheshire Cat can disappear gradually, leaving only its grin.",
    "Alice changes size by eating and drinking magical items.",
]

print("📋 測試資料集:")
print("=" * 60)
for i, (q, a) in enumerate(zip(test_questions, ground_truths)):
    print(f"\n{i+1}. Q: {q}")
    print(f"   A: {a[:60]}...")



📋 測試資料集:

1. Q: How did Alice fall into Wonderland?
   A: Alice fell down a rabbit hole while following a white rabbit...

2. Q: Who is the Queen of Hearts?
   A: The Queen of Hearts is a tyrannical ruler who orders executi...

3. Q: What happened at the Mad Tea Party?
   A: Alice attended a tea party with the Mad Hatter and March Har...

4. Q: What is special about the Cheshire Cat?
   A: The Cheshire Cat can disappear gradually, leaving only its g...

5. Q: How does Alice change size in the story?
   A: Alice changes size by eating and drinking magical items....


---
## 📊 Part 3: 執行 RAG 並收集評估資料

In [14]:
try:
    # Replace 'llm' with your LLM object (Ollama, ChatOpenAI, etc.)
    print(f"LLM Response: {llm.invoke('Hi')}")
    print("✅ LLM Connection: Success")
except Exception as e:
    print(f"❌ LLM Connection: Failed\n{e}")

LLM Response: Hello! How can I assist you today? 😊
✅ LLM Connection: Success


In [15]:
# Replace 'vectorstore' with whatever you named your retriever's source
try:
    test_search = vectorstore.similarity_search("test", k=1)
    print("✅ Vector DB Connection: Success")
except Exception as e:
    print(f"❌ Vector DB Connection: Failed\n{e}")

✅ Vector DB Connection: Success


In [16]:
# 執行 RAG 並收集結果
print("🔄 執行 RAG 系統...")
print("=" * 60)

rag_results = []    
for i, question in enumerate(test_questions):
    print(f"\n處理問題 {i+1}/{len(test_questions)}: {question[:40]}...")
    
    result = rag_chain.invoke({"query": question})
    
    # 收集資料
    contexts = [doc.page_content for doc in result["source_documents"]]
    
    rag_results.append({
        "question": question,
        "answer": result["result"],
        "contexts": contexts,
        "ground_truth": ground_truths[i]
    })

print("\n✅ RAG 執行完成！")



🔄 執行 RAG 系統...

處理問題 1/5: How did Alice fall into Wonderland?...

處理問題 2/5: Who is the Queen of Hearts?...

處理問題 3/5: What happened at the Mad Tea Party?...

處理問題 4/5: What is special about the Cheshire Cat?...

處理問題 5/5: How does Alice change size in the story?...

✅ RAG 執行完成！


In [32]:
---
## 📈 Part 4: RAGAS 評估

SyntaxError: invalid syntax (2820452174.py, line 1)

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

# 準備 RAGAS 資料集
eval_data = {
    "question": [r["question"] for r in rag_results],
    "answer": [r["answer"] for r in rag_results],
    "contexts": [r["contexts"] for r in rag_results],
    "ground_truth": [r["ground_truth"] for r in rag_results],
}

dataset = Dataset.from_dict(eval_data)

print("📊 執行 RAGAS 評估...")
print("=" * 60)

# 執行評估 (使用本地 LLM 作為評估模型)
# 注意：RAGAS 默認使用 OpenAI，這裡我們手動計算簡化版指標
# 完整版需要配置 LLM

# 簡化版評估 - 手動計算
import numpy as np

def simple_relevancy_score(question, answer):
    """簡化的相關性評分"""
    q_words = set(question.lower().split())
    a_words = set(answer.lower().split())
    overlap = len(q_words & a_words)
    return min(overlap / max(len(q_words), 1), 1.0)

def simple_context_score(answer, contexts):
    """簡化的上下文利用評分"""
    context_text = " ".join(contexts).lower()
    answer_words = answer.lower().split()
    found = sum(1 for w in answer_words if w in context_text)
    return found / max(len(answer_words), 1)

# 計算每個問題的分數
scores = []
for r in rag_results:
    relevancy = simple_relevancy_score(r["question"], r["answer"])
    context_use = simple_context_score(r["answer"], r["contexts"])
    scores.append({
        "question": r["question"][:30] + "...",
        "relevancy": relevancy,
        "context_use": context_use
    })

import pandas as pd
df_scores = pd.DataFrame(scores)
print("\n📊 評估結果:")
print(df_scores.to_string(index=False))

print(f"\n📈 平均分數:")
print(f"   Answer Relevancy: {df_scores['relevancy'].mean():.3f}")
print(f"   Context Utilization: {df_scores['context_use'].mean():.3f}")



In [ ]:
---
## 🎯 Lab 05 總結 & 課程總結

### RAGAS 核心指標解讀

| 指標 | 含義 | 改善方向 |
|------|------|----------|
| Faithfulness | 回答忠實度 | 改善 prompt、減少幻覺 |
| Answer Relevancy | 答案相關性 | 優化 retrieval |
| Context Precision | 檢索精確度 | 調整 chunk size、reranking |
| Context Recall | 檢索召回率 | Hybrid search、multi-query |

### 五堂 Lab 完整回顧

| Lab | 主題 | 核心技能 |
|-----|------|----------|
| Lab 01 | Hello World RAG | 基礎 RAG pipeline |
| Lab 02 | Vector DB | ChromaDB, FAISS, Qdrant |
| Lab 03 | Hybrid Search | BM25 + Vector + Ensemble |
| Lab 04 | Advanced RAG | Multi-Query, Reranking |
| Lab 05 | Evaluation | RAGAS 評估系統 |

### 🎉 恭喜完成所有 Lab！

您現在已經掌握了：
- ✅ RAG 系統的完整建構流程
- ✅ 多種 Vector Database 的操作
- ✅ 混合搜索策略
- ✅ 進階 RAG 技術
- ✅ 使用 RAGAS 評估系統品質

Successfully downloaded 'data/alice_in_wonderland.txt'.


## Generate Lab 1: Hello World RAG

### Subtask:
Provide the Markdown and Code cells for the first RAG lab, covering Loading, Splitting, Embedding, and Retrieval.


## Lab 1: Hello World RAG

This lab will walk you through the fundamental components of a Retrieval Augmented Generation (RAG) system, often referred to as a 'Hello World' RAG. We will cover the core steps required to prepare your data for retrieval and then perform a basic retrieval operation.

### Steps Covered:
1.  **Document Loading**: How to load raw text documents into a format suitable for processing.
2.  **Text Splitting**: Breaking down large documents into smaller, manageable chunks to improve retrieval relevance.
3.  **Embedding Generation**: Converting text chunks into numerical vector representations (embeddings) that capture their semantic meaning.
4.  **Retrieval**: Using a vector store to find the most relevant document chunks based on a query's embedding.

**Reasoning**:
The next step, as per the instructions, is to load the 'alice_in_wonderland.txt' document using `TextLoader` from `langchain_community.document_loaders`, ensuring the correct file path and encoding are specified.



In [17]:
from langchain_community.document_loaders import TextLoader

# Load the document
loader = TextLoader('data/alice_in_wonderland.txt', encoding='utf-8')
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")
print(f"First 200 characters of the document:\n{documents[0].page_content[:200]}")

Loaded 1 document(s).
First 200 characters of the document:
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho


**Reasoning**:
The previous code failed because the 'langchain_community' module was not found. This indicates that 'langchain-community' needs to be explicitly installed. Therefore, the immediate next step is to install the missing package.



In [6]:
pip install --quiet langchain-community
print("Installed langchain-community.")

SyntaxError: invalid syntax (ipython-input-3686645823.py, line 1)

**Reasoning**:
The previous command produced a SyntaxError because `pip install` is a shell command and needs to be prefixed with `!` when run in a Jupyter/Colab environment. The fix is to add `!` to the `pip install` command.



In [7]:
get_ipython().system('pip install --quiet langchain-community')
print("Installed langchain-community.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires ope

**Reasoning**:
Since `langchain-community` has been successfully installed, the next step is to re-run the code cell that loads the 'alice_in_wonderland.txt' document using `TextLoader` from `langchain_community.document_loaders`, which was the original intent after the lab introduction.



In [18]:
from langchain_community.document_loaders import TextLoader

# Load the document
loader = TextLoader('data/alice_in_wonderland.txt', encoding='utf-8')
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")
print(f"First 200 characters of the document:\n{documents[0].page_content[:200]}")

Loaded 1 document(s).
First 200 characters of the document:
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho


**Reasoning**:
Following the instructions, the next step is to split the loaded `documents` into smaller, manageable chunks using `RecursiveCharacterTextSplitter` from `langchain.text_splitter` with specified `chunk_size` and `chunk_overlap`.



In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

# Split the documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document(s) into {len(chunks)} chunks.")
print(f"First chunk (first 200 chars):\n{chunks[0].page_content[:200]}")
print(f"First chunk metadata:\n{chunks[0].metadata}")

Split 1 document(s) into 191 chunks.
First chunk (first 200 chars):
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho
First chunk metadata:
{'source': 'data/alice_in_wonderland.txt', 'start_index': 0}


**Reasoning**:
The previous code failed due to an `SyntaxError: unterminated f-string literal`. This occurs because a newline character was present directly within a single-line f-string. To resolve this, the f-strings for printing the chunk content and metadata will be converted to triple-quoted f-strings, allowing them to span multiple lines correctly.



In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

# Split the documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document(s) into {len(chunks)} chunks.")
print(f"First chunk (first 200 chars):\n{chunks[0].page_content[:200]}")
print(f"First chunk metadata:\n{chunks[0].metadata}")

Split 1 document(s) into 191 chunks.
First chunk (first 200 chars):
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho
First chunk metadata:
{'source': 'data/alice_in_wonderland.txt', 'start_index': 0}


**Reasoning**:
The previous code failed with a `ModuleNotFoundError: No module named 'langchain.text_splitter'`. This indicates that the `RecursiveCharacterTextSplitter` class, which was previously imported from `langchain.text_splitter`, is now located in a different module or requires a separate installation. The correct package to import it from is `langchain_text_splitters`. Therefore, the immediate next step is to install this missing package.



In [11]:
get_ipython().system('pip install --quiet langchain-text-splitters')
print("Installed langchain-text-splitters.")

Installed langchain-text-splitters.


**Reasoning**:
Since `langchain-text-splitters` has been successfully installed, the next step is to re-run the code cell that splits the document into chunks using `RecursiveCharacterTextSplitter`, this time importing it from `langchain_text_splitters` as previously identified.



In [61]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 深度優化切分器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,         # 縮小 size 讓資訊更聚焦
    chunk_overlap=200,      # 增加重疊度 (約 30%)，確保上下文連貫
    length_function=len,
    add_start_index=True,
    # 改進分隔符：加入對話標點，並優化切分順序
    separators=["\n\n", "\n", "。", "？", "！", ".", "?", "!", " ", ""]
)

# 2. 執行切分
chunks = text_splitter.split_documents(documents)

# 3. 智慧 Metadata 注入：自動辨識章節
current_chapter = "Prologue"
updated_chunks = []

for i, chunk in enumerate(chunks):
    # 利用正則表達式尋找文本中的 CHAPTER X 字樣
    # 小說原文格式通常是 CHAPTER I. , CHAPTER II. 等
    chapter_match = re.search(r"CHAPTER [IVXLCDM]+\.", chunk.page_content)
    if chapter_match:
        current_chapter = chapter_match.group()
    
    # 注入更有意義的元數據
    chunk.metadata["id"] = f"alice_chunk_{i}"
    chunk.metadata["chapter"] = current_chapter
    # 這裡可以加入一個 'chunk_seq'，幫助評估 Precision 時確認排序
    chunk.metadata["chunk_seq"] = i 
    
    updated_chunks.append(chunk)

print(f"✅ 優化完成：切分為 {len(updated_chunks)} 個區塊。")
print(f"📊 章節分佈範例：{updated_chunks[100].metadata['chapter']}")

✅ 優化完成：切分為 294 個區塊。
📊 章節分佈範例：CHAPTER V.


In [ ]:
'''
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

# Split the documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document(s) into {len(chunks)} chunks.")
print(f"First chunk (first 200 chars):\n{chunks[0].page_content[:200]}")
print(f"First chunk metadata:\n{chunks[0].metadata}"
'''

Split 1 document(s) into 191 chunks.
First chunk (first 200 chars):
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Ho
First chunk metadata:
{'source': 'data/alice_in_wonderland.txt', 'start_index': 0}


**Reasoning**:
Following the instructions, the next step is to initialize an embedding model using `SentenceTransformerEmbeddings` from `langchain_community.embeddings` and generate embeddings for the document chunks. The model 'all-MiniLM-L6-v2' is specified for this purpose.



In [63]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# Initialize the embedding model
embeddings_model = SentenceTransformerEmbeddings(model_name="voyageai/voyage-4-nano")

print("Embedding model initialized successfully.")

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 1974.01it/s]
Qwen3Model LOAD REPORT from: voyageai/voyage-4-nano
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model initialized successfully.


**Reasoning**:
The embedding model has been successfully initialized. The next step, as per the instructions, is to create a vector store (e.g., using `Chroma` from `langchain_community.vectorstores`) from the generated embeddings and document chunks, and then perform a similarity search using a sample query.



In [64]:
from langchain_community.vectorstores import Chroma

# Create a vector store from the chunks and embeddings
# This will embed the chunks and store them in a ChromaDB instance
vectorstore = Chroma.from_documents(chunks, embeddings_model)

print(f"Vector store created with {vectorstore._collection.count()} items.")

# Perform a similarity search (retrieval)
query = "What is Alice's adventure about?"
docs = vectorstore.similarity_search(query)

print(f"\nRetrieved {len(docs)} documents for the query: '{query}'")
print("--------------------------------------------------")
for i, doc in enumerate(docs):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}):\n{doc.page_content[:500]}...\n")

Vector store created with 1738 items.

Retrieved 4 documents for the query: 'What is Alice's adventure about?'
--------------------------------------------------
Document 1 (Source: data/alice_in_wonderland.txt, Start Index: None):
Alice watched the White Rabbit as he fumbled over the list, feeling
very curious to see what the next witness would be like, “—for they
haven’t got much evidence _yet_,” she said to herself. Imagine her
surprise, when the White Rabbit read out, at the top of his shrill
little voice, the name “Alice!”




CHAPTER XII.
Alice’s Evidence...

Document 2 (Source: data/alice_in_wonderland.txt, Start Index: None):
Alice watched the White Rabbit as he fumbled over the list, feeling
very curious to see what the next witness would be like, “—for they
haven’t got much evidence _yet_,” she said to herself. Imagine her
surprise, when the White Rabbit read out, at the top of his shrill
little voice, the name “Alice!”




CHAPTER XII.
Alice’s Evidence...

Document 3 (Source

## Generate Lab 2: Vector DB Deep Dive

### Subtask:
Provide the Markdown and Code cells for resource estimation and metadata filtering in Vector Databases.


## Lab 2: Vector DB Deep Dive

This lab delves deeper into the practical aspects of working with Vector Databases, focusing on crucial considerations for building robust and efficient RAG systems. Understanding these concepts is vital for optimizing performance and managing resources effectively.

### Topics Covered:
1.  **Resource Estimation**: How to estimate the memory requirements for storing embeddings in a vector database, based on factors like the number of chunks and embedding dimensions.
2.  **Metadata Filtering**: Enhancing retrieval accuracy and efficiency by leveraging metadata to narrow down search results, allowing for more precise contextual retrieval.

### Resource Estimation for Vector Databases

When building RAG systems, it's crucial to understand the resource implications of storing embeddings. Vector databases store high-dimensional numerical representations of text, and their memory footprint can grow significantly with the volume of data.

Key factors influencing resource estimation:
-   **Number of Documents/Chunks**: Each chunk of text will be converted into a vector. More chunks mean more vectors to store.
-   **Embedding Dimension**: The size of each vector (the number of dimensions) directly impacts memory usage. Larger dimensions lead to larger vectors.
-   **Data Type**: Embeddings are typically stored as floating-point numbers. `float32` (4 bytes per number) is common, but `float16` (2 bytes) can halve memory usage at the cost of some precision, and `float64` (8 bytes) offers higher precision but doubles memory.

To estimate the memory required, you can use the formula: `Number of Chunks * Embedding Dimension * Bytes per float`.

**Reasoning**:
Following the instructions, the next step is to create a Code cell to calculate the estimated memory usage for the existing vector store, using the number of chunks and the embedding dimension with a float32 assumption.



In [65]:
import sys

# Get the number of chunks
num_chunks = len(chunks)

# Determine the embedding dimension
# We'll embed a sample query and get the length of the resulting vector
sample_embedding = embeddings_model.embed_query('test')
embedding_dimension = len(sample_embedding)

# Assume float32 for embedding storage (4 bytes per dimension)
bytes_per_float = 4

# Calculate total estimated memory
estimated_memory_bytes = num_chunks * embedding_dimension * bytes_per_float
estimated_memory_mb = estimated_memory_bytes / (1024**2)
estimated_memory_gb = estimated_memory_bytes / (1024**3)

print(f"Number of chunks: {num_chunks}")
print(f"Embedding dimension: {embedding_dimension}")
print(f"Assumed bytes per float: {bytes_per_float}")
print(f"Estimated memory for embeddings: {estimated_memory_bytes:.2f} bytes")
print(f"Estimated memory for embeddings: {estimated_memory_mb:.2f} MB")
print(f"Estimated memory for embeddings: {estimated_memory_gb:.2f} GB")

Number of chunks: 294
Embedding dimension: 1024
Assumed bytes per float: 4
Estimated memory for embeddings: 1204224.00 bytes
Estimated memory for embeddings: 1.15 MB
Estimated memory for embeddings: 0.00 GB


### Metadata Filtering

Metadata filtering is a powerful technique to enhance the precision and relevance of retrieval in RAG systems. While vector similarity search is excellent for semantic matching, it doesn't always capture all the nuances or specific requirements of a query.

Metadata refers to additional information or attributes associated with each document or chunk. This can include:
-   **Categorical data**: document type (e.g., 'Introduction', 'Main Body', 'Appendix'), author, publication year, topic.
-   **Numerical data**: `start_index` (position in the original document), page number, section ID.

By leveraging metadata filters, you can:
-   **Narrow down search space**: Restrict similarity search to only chunks that match specific criteria, significantly improving efficiency and relevance.
-   **Improve precision**: Ensure retrieved documents meet specific structural or contextual requirements (e.g., "find answers about Alice's adventures only from the 'Main Body' section").
-   **Enhance control**: Provide more granular control over what information is presented to the language model.

Vector databases like Chroma support robust metadata filtering, allowing you to combine semantic similarity with structured filtering conditions.

**Reasoning**:
Following the instructions, the next step is to create a Code cell to demonstrate metadata filtering. This involves preparing chunks with new metadata, re-creating the vector store, and then performing both numerical and categorical filtered retrievals.



In [78]:
from langchain_core.documents import Document

# a. Prepare Chunks with Metadata
updated_chunks = []
for i, chunk in enumerate(chunks):
    new_metadata = chunk.metadata.copy()
    if i < 10:
        new_metadata['doc_part'] = 'Introduction'
    elif i < 100:
        new_metadata['doc_part'] = 'Main Body'
    else:
        new_metadata['doc_part'] = 'Appendix'
    updated_chunks.append(Document(page_content=chunk.page_content, metadata=new_metadata))

print(f"Prepared {len(updated_chunks)} chunks with updated metadata.")
print(f"Example of updated metadata for first chunk: {updated_chunks[0].metadata}")
print(f"Example of updated metadata for chunk 15: {updated_chunks[15].metadata}")
print(f"Example of updated metadata for chunk 105: {updated_chunks[105].metadata}")

# b. Re-create Vector Store with updated chunks and embeddings
# We'll use a new Chroma instance to demonstrate the filtering clearly
vectorstore_filtered = Chroma.from_documents(updated_chunks, embeddings_model, collection_name="alice_filtered_data")

print(f"Vector store with filtered metadata created with {vectorstore_filtered._collection.count()} items.")

# c. Perform Filtered Retrieval (Numerical Metadata)
query_num = "Who is the main character and what is her story about?"
# Filter for chunks that start after index 50000
#docs_num_filtered = vectorstore_filtered.similarity_search(query_num, k=3, where={"start_index": {"$gt": 50000}})
docs_num_filtered = vectorstore_filtered.similarity_search(
    query_num, 
    k=3, 
    filter={"start_index": {"$gt": 50000}} # LangChain uses 'filter'
)
print(f"\nRetrieved {len(docs_num_filtered)} documents for query '{query_num}' with numerical filter (start_index > 50000):")
print("--------------------------------------------------")
for i, doc in enumerate(docs_num_filtered):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}, Doc Part: {doc.metadata.get('doc_part')}):\n{doc.page_content[:200]}...\n")

# d. Perform Filtered Retrieval (Categorical Metadata)
query_cat = "Who does Alice meet at a tea party?"
# Filter for chunks only from 'Main Body'
#docs_cat_filtered = vectorstore_filtered.similarity_search(query_cat, k=3, where={"doc_part": "Main Body"})
docs_cat_filtered = vectorstore_filtered.similarity_search(
    query_cat, 
    k=3, 
    filter={"doc_part": "Main Body"} # LangChain uses 'filter'
)
print(f"\nRetrieved {len(docs_cat_filtered)} documents for query '{query_cat}' with categorical filter (doc_part = 'Main Body'):")
print("--------------------------------------------------")
for i, doc in enumerate(docs_cat_filtered):
    print(f"Document {i+1} (Source: {doc.metadata.get('source')}, Start Index: {doc.metadata.get('start_index')}, Doc Part: {doc.metadata.get('doc_part')}):\n{doc.page_content[:200]}...\n")

Prepared 294 chunks with updated metadata.
Example of updated metadata for first chunk: {'source': 'data/alice_in_wonderland.txt', 'start_index': 0, 'id': 'alice_chunk_0', 'chapter': 'CHAPTER I.', 'chunk_seq': 0, 'doc_part': 'Introduction'}
Example of updated metadata for chunk 15: {'source': 'data/alice_in_wonderland.txt', 'start_index': 6499, 'id': 'alice_chunk_15', 'chapter': 'CHAPTER I.', 'chunk_seq': 15, 'doc_part': 'Main Body'}
Example of updated metadata for chunk 105: {'source': 'data/alice_in_wonderland.txt', 'start_index': 49513, 'id': 'alice_chunk_105', 'chapter': 'CHAPTER V.', 'chunk_seq': 105, 'doc_part': 'Appendix'}
Vector store with filtered metadata created with 1832 items.

Retrieved 3 documents for query 'Who is the main character and what is her story about?' with numerical filter (start_index > 50000):
--------------------------------------------------
Document 1 (Source: data/alice_in_wonderland.txt, Start Index: 67420, Doc Part: Appendix):
Alice was just beginning

In [97]:
from langchain_community.vectorstores import Chroma

print(f"正在將 {len(updated_chunks)} 個區塊存入向量資料庫...")

vectorstore_final = Chroma.from_documents(
    documents=updated_chunks, 
    embedding=embeddings_model,
    collection_name="alice_final_v2"
)

print("✅ vectorstore_final 已成功建立！")

正在將 294 個區塊存入向量資料庫...
✅ vectorstore_final 已成功建立！


In [98]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. 向量相似度工具 (沿用你的設定)
def get_vec(text):
    return np.array(embeddings_model.embed_query(text)).reshape(1, -1)

def vector_sim(text1, text2):
    if not text1 or not text2: return 0.0
    v1 = get_vec(str(text1))
    v2 = get_vec(str(text2))
    return float(cosine_similarity(v1, v2)[0][0])

# 2. 準備測試問題 (確保與你的章節標籤 CHAPTER X. 格式一致)
test_questions = [
    {"question": "How did Alice fall into Wonderland?", "ground_truth": "Alice fell down a rabbit hole while following a white rabbit.", "chapter": "CHAPTER I."},
    {"question": "What did Alice find on the table?", "ground_truth": "A tiny golden key and a bottle labeled DRINK ME.", "chapter": "CHAPTER I."},
    {"question": "Who is the Queen of Hearts?", "ground_truth": "The Queen of Hearts is a tyrannical ruler who orders executions.", "chapter": "CHAPTER VIII."},
    {"question": "What happened at the Mad Tea Party?", "ground_truth": "Alice met the Hatter and March Hare and had a confusing conversation.", "chapter": "CHAPTER VII."}
]

# 3. 執行評估流程
final_results = []

print("📊 執行優化後的 RAGAS 模擬評估...")
print("=" * 70)

for item in test_questions:
    q = item["question"]
    gt = item["ground_truth"]
    target_chap = item["chapter"]
    
    # --- 檢索階段 (使用你最新的 vectorstore_final) ---
    search_kwargs = {"k": 3}
    if target_chap:
        search_kwargs["filter"] = {"chapter": target_chap}
    
    retrieved_docs = vectorstore_final.similarity_search(q, k=5, filter={"chapter": target_chap})
    
    # Fallback 機制
    if not retrieved_docs:
        retrieved_docs = vectorstore_final.similarity_search(q, k=5 )
        
    # 合併上下文用於計算 Faithfulness 和 Recall
    contexts_text = " ".join([d.page_content for d in retrieved_docs])
    
    # --- 生成階段 (此處應代入你的 LLM invoke 邏輯) ---
    # 範例：假設 LLM 產出的答案與 GT 語義接近
    answer = f"Based on the documents, {gt}" 
    
    # --- 指標計算 ---
    # 1. Faithfulness: 回答與檢索內容的語義重疊
    faith = vector_sim(answer, contexts_text)
    
    # 2. Answer Relevancy: 回答與問題的直接相關性
    rel = vector_sim(answer, q)
    
    # 3. Context Precision: 分開計算每個 doc 的貢獻，看排序是否精確
    doc_scores = [vector_sim(doc.page_content, gt) for doc in retrieved_docs]
    if doc_scores:
        current_weights = [1.0, 0.8, 0.6, 0.4, 0.2][:len(doc_scores)]
        prec = np.average(np.array(doc_scores, dtype=float), weights=current_weights)
    else:
        prec = 0.0
    
    # 4. Context Recall: 整體 Context 與 Ground Truth 的覆蓋率
    recall = vector_sim(contexts_text, gt)

    final_results.append({
        "Chapter": target_chap,
        "Question": q[:20] + "...",
        "Faithfulness": round(faith, 3),
        "Relevancy": round(rel, 3),
        "Precision": round(prec, 3),
        "Recall": round(recall, 3)
    })

# 4. 輸出最終報表
df_final = pd.DataFrame(final_results)
print(df_final.to_string(index=False))

print("-" * 70)
print(f"📈 優化後平均分數:")
print(f"   Faithfulness:      {df_final['Faithfulness'].mean():.3f}")
print(f"   Answer Relevancy:   {df_final['Relevancy'].mean():.3f}")
print(f"   Context Precision: {df_final['Precision'].mean():.3f}")
print(f"   Context Recall:    {df_final['Recall'].mean():.3f}")


📊 執行優化後的 RAGAS 模擬評估...
      Chapter                Question  Faithfulness  Relevancy  Precision  Recall
   CHAPTER I. How did Alice fall i...         0.673      0.732      0.726   0.696
   CHAPTER I. What did Alice find ...         0.481      0.503      0.432   0.474
CHAPTER VIII. Who is the Queen of ...         0.498      0.737      0.551   0.512
 CHAPTER VII. What happened at the...         0.519      0.533      0.561   0.502
----------------------------------------------------------------------
📈 優化後平均分數:
   Faithfulness:      0.543
   Answer Relevancy:   0.626
   Context Precision: 0.568
   Context Recall:    0.546


In [ ]:
# --- 1. 重新執行評估流程 (確保存入 GT 與 Answer) ---
final_results = []

print("📊 正在產生含對照組的評估報表...")

for item in test_questions:
    q = item["question"]
    gt = item["ground_truth"]
    target_chap = item["chapter"]
    
    # 檢索
    retrieved_docs = vectorstore_final.similarity_s earch(q, k=5, filter={"chapter": target_chap})
    if not retrieved_docs:
        retrieved_docs = vectorstore_final.similarity_search(q, k=5)
        
    contexts_text = " ".join([d.page_content for d in retrieved_docs])
    
    # 模擬 LLM 回答 (在此處確保變數名稱為 answer)
    answer = f"According to the text, {gt}" 
    
    # 指標計算
    faith = vector_sim(answer, contexts_text)
    rel = vector_sim(answer, q)
    doc_scores = [vector_sim(doc.page_content, gt) for doc in retrieved_docs]
    prec = np.average(np.array(doc_scores, dtype=float), weights=[1, 0.8, 0.6, 0.4, 0.2][:len(doc_scores)]) if doc_scores else 0
    recall = vector_sim(contexts_text, gt)
    f1 = 2 * (prec * recall) / (prec + recall) if (prec + recall) > 0 else 0

    # 重點：這裡必須包含 'GT' 與 'Answer' 鍵值
    final_results.append({
        "Chapter": target_chap,
        "Question": q,
        "GT": gt,             # 標準答案
        "Answer": answer,     # 模型回答
        "Faith": round(faith, 3),
        "Rel": round(rel, 3),
        "Prec": round(prec, 3),
        "Rec": round(recall, 3),
        "F1": round(f1, 3)
    })

df_final = pd.DataFrame(final_results)

# --- 2. 顯示深藍對比報表 ---
def style_rag_final_comparison(df):
    styled = df.style.hide(axis="index")
    metrics = ['Faith', 'Rel', 'Prec', 'Rec', 'F1']
    
    # 設定背景顏色梯度
    styled = styled.background_gradient(cmap='YlGnBu', subset=metrics, vmin=0.3, vmax=0.8)
    
    # 全域樣式 (全置中)
    styled = styled.set_properties(**{
        'text-align': 'center',
        'vertical-align': 'middle',
        'padding': '12px',
        'border': '1px solid #30475e',
        'background-color': '#1a1a2e',
        'color': '#eeeeee',
        'white-space': 'normal',
        'word-break': 'break-word'
    })
    
    # 針對文字欄位設定寬度與顏色
    styled = styled.set_properties(subset=['Question', 'GT', 'Answer'], **{'min-width': '250px'})
    styled = styled.set_properties(subset=['GT'], **{'color': '#f1c40f'})      # 黃色 GT
    styled = styled.set_properties(subset=['Answer'], **{'color': '#80ed99'})  # 綠色 Answer
    
    header_style = {
        'selector': 'th',
        'props': [('background-color', '#162447'), ('color', '#00fff5'), ('text-align', 'center'), ('padding', '15px')]
    }
    
    return styled.format("{:.3f}", subset=metrics).set_table_styles([header_style])

display(style_rag_final_comparison(df_final))

📊 正在產生含對照組的評估報表...


Chapter,Question,GT,Answer,Faith,Rel,Prec,Rec,F1
CHAPTER I.,How did Alice fall into Wonderland?,Alice fell down a rabbit hole while following a white rabbit.,"According to the text, Alice fell down a rabbit hole while following a white rabbit.",0.698,0.725,0.726,0.696,0.711
CHAPTER I.,What did Alice find on the table?,A tiny golden key and a bottle labeled DRINK ME.,"According to the text, A tiny golden key and a bottle labeled DRINK ME.",0.494,0.480,0.432,0.474,0.452
CHAPTER VIII.,Who is the Queen of Hearts?,The Queen of Hearts is a tyrannical ruler who orders executions.,"According to the text, The Queen of Hearts is a tyrannical ruler who orders executions.",0.541,0.704,0.551,0.512,0.531
CHAPTER VII.,What happened at the Mad Tea Party?,Alice met the Hatter and March Hare and had a confusing conversation.,"According to the text, Alice met the Hatter and March Hare and had a confusing conversation.",0.545,0.538,0.561,0.502,0.530


In [105]:
import re

# 1. 定義清洗函數 (去除開場白)
def clean_answer(text):
    # 定義要過濾的開場白列表 (不分大小寫)
    prefixes = [
        r"^according to the text,?\s*",
        r"^based on the documents,?\s*",
        r"^according to the provided information,?\s*",
        r"^the text states that\s*"
    ]
    cleaned = text
    for p in prefixes:
        cleaned = re.sub(p, "", cleaned, flags=re.IGNORECASE)
    return cleaned.strip()

# 2. 重新執行評估流程
final_results = []

for item in test_questions:
    q = item["question"]
    gt = item["ground_truth"]
    target_chap = item["chapter"]
    
    # 檢索
    retrieved_docs = vectorstore_final.similarity_search(q, k=5, filter={"chapter": target_chap})
    contexts_text = " ".join([d.page_content for d in retrieved_docs])
    
    # 原始回答
    raw_answer = f"According to the text, {gt}" 
    
    # --- 關鍵步驟：清洗回答 ---
    # 去除贅字後再進行向量計算
    processed_answer = clean_answer(raw_answer)
    
    # 指標計算 (使用 processed_answer)
    faith = vector_sim(processed_answer, contexts_text)
    rel = vector_sim(processed_answer, q)
    
    # ... (其餘 Prec, Rec, F1 計算邏輯不變) ...
    doc_scores = [vector_sim(doc.page_content, gt) for doc in retrieved_docs]
    prec = np.average(np.array(doc_scores, dtype=float), weights=[1, 0.8, 0.6, 0.4, 0.2][:len(doc_scores)])
    recall = vector_sim(contexts_text, gt)
    f1 = 2 * (prec * recall) / (prec + recall) if (prec + recall) > 0 else 0

    final_results.append({
        "Chapter": target_chap,
        "Question": q,
        "GT": gt,
        "Answer": processed_answer, # 存入清洗後的乾淨回答
        "Faith": round(faith, 3),
        "Rel": round(rel, 3),
        "Prec": round(prec, 3),
        "Rec": round(recall, 3),
        "F1": round(f1, 3)
    })

df_final = pd.DataFrame(final_results)

# 顯示報表 (沿用之前的深藍置中樣式)
display(style_rag_final_comparison(df_final))

Chapter,Question,GT,Answer,Faith,Rel,Prec,Rec,F1
CHAPTER I.,How did Alice fall into Wonderland?,Alice fell down a rabbit hole while following a white rabbit.,Alice fell down a rabbit hole while following a white rabbit.,0.696,0.673,0.726,0.696,0.711
CHAPTER I.,What did Alice find on the table?,A tiny golden key and a bottle labeled DRINK ME.,A tiny golden key and a bottle labeled DRINK ME.,0.474,0.426,0.432,0.474,0.452
CHAPTER VIII.,Who is the Queen of Hearts?,The Queen of Hearts is a tyrannical ruler who orders executions.,The Queen of Hearts is a tyrannical ruler who orders executions.,0.512,0.742,0.551,0.512,0.531
CHAPTER VII.,What happened at the Mad Tea Party?,Alice met the Hatter and March Hare and had a confusing conversation.,Alice met the Hatter and March Hare and had a confusing conversation.,0.502,0.582,0.561,0.502,0.530


In [53]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

# 準備 RAGAS 資料集
eval_data = {
    "question": [r["question"] for r in rag_results],
    "answer": [r["answer"] for r in rag_results],
    "contexts": [r["contexts"] for r in rag_results],
    "ground_truth": [r["ground_truth"] for r in rag_results],
}

dataset = Dataset.from_dict(eval_data)

print("📊 執行 RAGAS 評估...")
print("=" * 60)

# 執行評估 (使用本地 LLM 作為評估模型)
# 注意：RAGAS 默認使用 OpenAI，這裡我們手動計算簡化版指標
# 完整版需要配置 LLM

# 簡化版評估 - 手動計算
import numpy as np

def simple_relevancy_score(question, answer):
    """簡化的相關性評分"""
    q_words = set(question.lower().split())
    a_words = set(answer.lower().split())
    overlap = len(q_words & a_words)
    return min(overlap / max(len(q_words), 1), 1.0)

def simple_context_score(answer, contexts):
    """簡化的上下文利用評分"""
    context_text = " ".join(contexts).lower()
    answer_words = answer.lower().split()
    found = sum(1 for w in answer_words if w in context_text)
    return found / max(len(answer_words), 1)

# 計算每個問題的分數
scores = []
for r in rag_results:
    relevancy = simple_relevancy_score(r["question"], r["answer"])
    context_use = simple_context_score(r["answer"], r["contexts"])
    scores.append({
        "question": r["question"][:30] + "...",
        "relevancy": relevancy,
        "context_use": context_use
    })

import pandas as pd
df_scores = pd.DataFrame(scores)
print("\n📊 評估結果:")
print(df_scores.to_string(index=False))

print(f"\n📈 平均分數:")
print(f"   Answer Relevancy: {df_scores['relevancy'].mean():.3f}")
print(f"   Context Utilization: {df_scores['context_use'].mean():.3f}")





📊 執行 RAGAS 評估...

📊 評估結果:
                         question  relevancy  context_use
How did Alice fall into Wonder...   0.333333     0.465517
   Who is the Queen of Hearts?...   0.666667     0.823529
What happened at the Mad Tea P...   0.428571     0.653543
What is special about the Ches...   0.571429     0.319444
How does Alice change size in ...   0.375000     0.562500

📈 平均分數:
   Answer Relevancy: 0.475
   Context Utilization: 0.565


/tmp/ipykernel_9521/2037705194.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_9521/2037705194.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_9521/2037705194.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_9521/2037705194.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed